In [1]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from datetime import datetime, timezone

app = FastAPI(title="Employee Attendance API")

employees = {"EMP001": "Ravi"}
attendance = {}

class PunchRequest(BaseModel):
    employee_id: str

@app.get("/")
def home():
    return {"message": "Employee Attendance API is running"}

@app.post("/attendance/punch-in")
def punch_in(request: PunchRequest):
    emp_id = request.employee_id

    if emp_id not in employees:
        raise HTTPException(404, "Employee not found")

    if emp_id in attendance and attendance[emp_id]["punch_out"] is None:
        raise HTTPException(400, "Already punched in")

    now = datetime.now(timezone.utc)
    attendance[emp_id] = {
        "punch_in": now,
        "punch_out": None,
        "working_hours": None
    }

    return {
        "message": "Punch-in successful",
        "employee_id": emp_id,
        "punch_in": now.isoformat()
    }

@app.post("/attendance/punch-out")
def punch_out(request: PunchRequest):
    emp_id = request.employee_id

    if emp_id not in attendance or attendance[emp_id]["punch_out"] is not None:
        raise HTTPException(400, "No active punch-in found")

    now = datetime.now(timezone.utc)
    record = attendance[emp_id]
    hours = round((now - record["punch_in"]).total_seconds() / 3600, 2)

    record["punch_out"] = now
    record["working_hours"] = hours

    return {
        "message": "Punch-out successful",
        "employee_id": emp_id,
        "punch_out": now.isoformat(),
        "working_hours": hours
    }

Writing main.py


In [ ]:
!uvicorn main:app --host 0.0.0.0 --port 8000

INFO:     Started server process [1287]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
%pip install fastapi uvicorn requests

In [ ]:
import subprocess
import sys
import time

server = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "main:app",
    "--host", "127.0.0.1",
    "--port", "8000"
])

time.sleep(3)

print("Server running:", server.poll() is None)

Server running: True


In [ ]:
server = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "main:app", "--host", "127.0.0.1", "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(3)
print(server.stdout.read() if server.poll() is not None else "Server started")

INFO:     Started server process [7556]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



In [ ]:
import requests

response = requests.get("http://127.0.0.1:8000/")
print(response.status_code)
print(response.json())

200
{'message': 'Employee Attendance API is running'}
